In [1]:
def carregar_arff(caminho):
    X = []
    y = []
    lendo_dados = False

    with open(caminho, 'r') as f:
        for linha in f:
            linha = linha.strip()

            if linha == "" or linha.startswith("%"):
                continue

            if linha.lower() == "@data":
                lendo_dados = True
                continue

            if lendo_dados:
                partes = linha.split(",")

                # última coluna = classe
                atributos = partes[:-1]
                classe = partes[-1]

                # converter números quando possível
                nova_linha = []
                for val in atributos:
                    try:
                        nova_linha.append(float(val))
                    except:
                        nova_linha.append(val)

                X.append(nova_linha)
                y.append(classe)

    return X, y

In [3]:
Xc, yc = carregar_arff("datasets/BNG(credit-g)_classificacao.arff")
Xr, yr = carregar_arff("datasets/file22f1627e4a960_regressao.arff")

In [8]:
def codificar_categoricos(X):
    mapas = [{} for _ in range(len(X[0]))]

    for j in range(len(X[0])):
        valor_id = 0
        for i in range(len(X)):
            val = X[i][j]

            if isinstance(val, str):
                if val not in mapas[j]:
                    mapas[j][val] = valor_id
                    valor_id += 1
                X[i][j] = mapas[j][val]

    return X

In [10]:
Xc_numerico = codificar_categoricos(Xc)
Xr_numerico = codificar_categoricos(Xr)

In [33]:
def amostra_estratificada(X, y, n_total):
    from collections import defaultdict
    import random

    grupos = defaultdict(list)

    for i in range(len(y)):
        grupos[y[i]].append(i)

    X_out, y_out = [], []

    for classe, indices in grupos.items():
        n_classe = int(len(indices) / len(y) * n_total)
        escolhidos = random.sample(indices, n_classe)

        for i in escolhidos:
            X_out.append(X[i])
            y_out.append(y[i])

    return X_out, y_out

In [36]:
Xc_numerico_novo, yc_novo = amostra_estratificada(Xc_numerico, yc, 10000)

len(Xc_numerico_novo) #9999

9999

In [12]:
def normalizar(X):
    mins = [min(col) for col in zip(*X)]
    maxs = [max(col) for col in zip(*X)]

    X_norm = []
    for linha in X:
        nova = []
        for i in range(len(linha)):
            if maxs[i] == mins[i]:
                nova.append(0)
            else:
                nova.append((linha[i] - mins[i]) / (maxs[i] - mins[i]))
        X_norm.append(nova)

    return X_norm

In [37]:
Xc_normalizado = normalizar(Xc_numerico_novo)
Xr_normalizado = normalizar(Xr_numerico)

funções usadas no kfold_knn

In [16]:
def criar_folds(X, y, k):
    tamanho = len(X)
    fold_size = tamanho // k

    folds = []

    for i in range(k):
        inicio = i * fold_size
        fim = inicio + fold_size

        X_test = X[inicio:fim]
        y_test = y[inicio:fim]

        X_train = X[:inicio] + X[fim:]
        y_train = y[:inicio] + y[fim:]

        folds.append((X_train, X_test, y_train, y_test))

    return folds

In [17]:
def knn_classificacao(X_train, y_train, x_teste, k, distancia_func):
    distancias = []

    # calcular distância de x_teste para todos
    for i in range(len(X_train)):
        d = distancia_func(x_teste, X_train[i])
        distancias.append((d, y_train[i]))

    # ordenar pela menor distância
    distancias.sort(key=lambda x: x[0])

    # pegar os k mais próximos
    vizinhos = distancias[:k]

    # contar classes
    contagem = {}
    for _, classe in vizinhos:
        if classe not in contagem:
            contagem[classe] = 0
        contagem[classe] += 1

    # retornar classe mais frequente
    return max(contagem, key=contagem.get)

Funções para calcular_metricas

In [18]:
def matriz_confusao(y_real, y_pred, positivo="good"):
    TP = FP = TN = FN = 0

    for i in range(len(y_real)):
        if y_real[i] == positivo and y_pred[i] == positivo:
            TP += 1
        elif y_real[i] != positivo and y_pred[i] == positivo:
            FP += 1
        elif y_real[i] != positivo and y_pred[i] != positivo:
            TN += 1
        elif y_real[i] == positivo and y_pred[i] != positivo:
            FN += 1

    return TP, FP, TN, FN

In [19]:
def precision(TP, FP):
    if TP + FP == 0:
        return 0
    return TP / (TP + FP)

In [20]:
def recall(TP, FN):
    if TP + FN == 0:
        return 0
    return TP / (TP + FN)

In [21]:
def f1_score(p, r):
    if p + r == 0:
        return 0
    return 2 * (p * r) / (p + r)

métricas regressão

In [1]:
def r2_score(y_real, y_pred):
    y_real = list(y_real)
    y_pred = list(y_pred)
    
    media_y = sum(y_real) / len(y_real)
    
    ss_res = 0  # erro
    ss_tot = 0  # variância
    
    for i in range(len(y_real)):
        ss_res += (y_real[i] - y_pred[i]) ** 2
        ss_tot += (y_real[i] - media_y) ** 2
    
    return 1 - (ss_res / ss_tot)

In [2]:
def r2_ajustado(y_real, y_pred, p):
    n = len(y_real)
    r2 = r2_score(y_real, y_pred)
    
    return 1 - (1 - r2) * ((n - 1) / (n - p - 1))

FIM - Funções para calcular_metricas

In [22]:
def calcular_metricas(y_real, y_pred):
    TP, FP, TN, FN = matriz_confusao(y_real, y_pred)

    p = precision(TP, FP)
    r = recall(TP, FN)
    f1 = f1_score(p, r)

    acc = (TP + TN) / (TP + TN + FP + FN)

    return acc, p, r, f1

In [3]:
def calcular_metricas_regressao(y_real, y_pred, num_features):
    r2 = r2_score(y_real, y_pred)
    r2_adj = r2_ajustado(y_real, y_pred, num_features)
    
    return r2, r2_adj

FIM - funções usadas no kfold_knn

In [28]:
from tqdm import tqdm

def kfold_knn(X, y, k_folds, k_vizinhos, distancia_func):
    folds = criar_folds(X, y, k_folds)

    resultados = []

    for X_train, X_test, y_train, y_test in tqdm(folds, desc="K-Fold"):
        preds = []

        for x in X_test:
            pred = knn_classificacao(X_train, y_train, x, k_vizinhos, distancia_func)
            preds.append(pred)

        acc, p, r, f1 = calcular_metricas(y_test, preds)
        resultados.append((acc, p, r, f1))

    return resultados

In [49]:
import math

def distancia_euclidiana(x, y):
    soma = 0
    for i in range(len(x)):
        soma += (x[i] - y[i]) ** 2
    return math.sqrt(soma)

def distancia_manhattan(x, y):
    soma = 0
    for i in range(len(x)):
        soma += abs(x[i] - y[i])
    return soma

In [44]:
X1 = Xc_normalizado
y1 = yc_novo

In [ ]:
#len(Xc) #1.000.000
#len(Xr_numerico) #10.692

9999

In [45]:
resultados1 = kfold_knn(
    X1, y1,
    k_folds=3,
    k_vizinhos=3,
    distancia_func=distancia_euclidiana
)

K-Fold: 100%|██████████| 3/3 [03:29<00:00, 69.89s/it]


In [46]:
import math

def media(lista):
    return sum(lista) / len(lista)

def desvio_padrao(lista):
    m = media(lista)
    soma = sum((x - m) ** 2 for x in lista)
    return math.sqrt(soma / len(lista))

In [47]:
def resumo_metricas(resultados):
    accs = [r[0] for r in resultados]
    ps = [r[1] for r in resultados]
    rs = [r[2] for r in resultados]
    f1s = [r[3] for r in resultados]

    print("Accuracy:", media(accs), "±", desvio_padrao(accs))
    print("Precision:", media(ps), "±", desvio_padrao(ps))
    print("Recall:", media(rs), "±", desvio_padrao(rs))
    print("F1:", media(f1s), "±", desvio_padrao(f1s))

In [4]:
def resumo_metricas_regressao(resultados):
    r2s = [r[0] for r in resultados]
    r2_adjs = [r[1] for r in resultados]

    print("R2:", media(r2s), "±", desvio_padrao(r2s))
    print("R2 Ajustado:", media(r2_adjs), "±", desvio_padrao(r2_adjs))

In [48]:
resumo_metricas(resultados1)

Accuracy: 0.5142514251425142 ± 0.29340919707892
Precision: 0.6997699769976998 ± 0.4245893703614393
Recall: 0.8144814481448145 ± 0.13118422804683735
F1: 0.619140261862282 ± 0.3100409873379654


In [50]:
resultados2 = kfold_knn(
    X1, y1,
    k_folds=3,
    k_vizinhos=3,
    distancia_func=distancia_manhattan
)

K-Fold: 100%|██████████| 3/3 [03:10<00:00, 63.61s/it]


In [51]:
resumo_metricas(resultados2)

Accuracy: 0.5188518851885189 ± 0.29666402044755985
Precision: 0.6997699769976998 ± 0.4245893703614393
Recall: 0.8190819081908192 ± 0.1279355045911496
F1: 0.6222313376711383 ± 0.3122274706631362
